In [10]:
import os
import pickle

import torch
import torch.nn.functional as F
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# this part of code is duplicated from the work of https://github.com/jlko/semantic_uncertainty

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

class BaseEntailment:
    def save_prediction_cache(self):
        pass

class EntailmentDeberta(BaseEntailment):
    def __init__(self):
        self.tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v2-xlarge-mnli", weights_only=False, truncation=True, padding=True)
        # self.tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-large")
        self.model = AutoModelForSequenceClassification.from_pretrained(
            "microsoft/deberta-v2-xlarge-mnli", weights_only=False).to(DEVICE)

    def check_implication(self, text1, text2, *args, **kwargs):
        # The model checks if text1 -> text2, i.e. if text2 follows from text1.
        # check_implication('The weather is good', 'The weather is good and I like you') --> 1
        # check_implication('The weather is good and I like you', 'The weather is good') --> 2

        torch.cuda.empty_cache()

        inputs = self.tokenizer(text1, text2, return_tensors="pt", max_length=150, truncation=True, padding=True).to(DEVICE)

        outputs = self.model(**inputs)
        
        logits = outputs.logits
        
        class_scores = F.softmax(logits, dim=1)

        # Deberta-mnli returns `neutral` and `entailment` classes at indices 1 and 2.
        largest_index = torch.argmax(class_scores)  # pylint: disable=no-member
        prediction = largest_index.cpu().item()
        if os.environ.get('DEBERTA_FULL_LOG', False):
            logging.info('Deberta Input: %s -> %s', text1, text2)
            logging.info('Deberta Prediction: %s', prediction)

        del inputs
        del outputs
        del logits

        torch.cuda.empty_cache()

        return prediction, class_scores

    def get_entailment(self, text1, text2, *args, **kwargs):
        _, softmax_probs = self.check_implication(text1, text2, *args, **kwargs)
        torch.cuda.empty_cache()
        _entail_probs = [softmax_probs[i][2].cpu().detach().numpy().astype('float').item() for i in range(len(softmax_probs))]
        del softmax_probs
        return _entail_probs

In [11]:
try:
    del judger
except:
    print('first judger!')

judger = EntailmentDeberta()

In [3]:
try:
    torch.cuda.empty_cache()
    del x
    del y
except:
    print('x, y haven\'t been created yet.')

x, y haven't been created yet.


In [4]:
torch.cuda.empty_cache()

In [5]:
import pickle as pkl
import pandas as pd

f = open(f'../../rag_utility/doc_dicts/nq_wiki_dict.pkl', 'rb')
doc_dict = pkl.load(f)
f.close()

res = pd.read_csv(f'../../rag_utility/res/mt5_nq_test.csv')

def get_sentence_splitter():
    from sentence_splitter import SentenceSplitter
    _splitter = SentenceSplitter(language='en')
    return _splitter

In [13]:
from tqdm import tqdm

_k = 16

splitter = get_sentence_splitter()

max_max_tokens = 0
for _qid in tqdm(res.qid.unique()):
    doc_texts = res[(res.qid == _qid) & (res['rank'] <_k)].docno.apply(lambda x: doc_dict[str(x)]).values
    sentences = []
    for doc_text in doc_texts:
        sentences += splitter.split(doc_text)
    for i in range(len(sentences)): # the last one don't need to be calculated
        max_tokens = len(judger.tokenizer(sentences[i])['input_ids'])
        # max_max_tokens = max(max_tokens, max_max_tokens)
        if(max_tokens > 150):
            print(_qid, sentences[i])
    if(int(_qid.split('_')[1]) % 100 == 0):
        print(max_max_tokens)
print(max_max_tokens)

  0%|          | 9/3610 [00:00<01:21, 44.13it/s]

0


  1%|▏         | 54/3610 [00:01<01:17, 45.78it/s]

test_50 Beyoncé – ""Single Ladies (Put a Ring on It)"" T.I. (featuring Rihanna) – ""Live Your Life"" Taylor Swift – ""You Belong with Me"" Lady Gaga – ""Poker Face"" Britney Spears – ""Womanizer"" Green Day – ""21 Guns"" Eminem – ""We Made You"" Matt & Kim – ""Lessons Learned"" Green Day – ""21 Guns"" (Director: Marc Webb) Beyoncé – ""Single Ladies (Put a Ring on It)"" (Choreographers: Frank Gatson and JaQuel Knight) Lady Gaga – ""Paparazzi"" (Special Effects: Chimney Pot) Lady Gaga – ""Paparazzi"" (Art Director: Jason Hamilton)


  3%|▎         | 110/3610 [00:02<01:12, 48.34it/s]

...

0


 99%|█████████▉| 3584/3610 [01:15<00:00, 46.73it/s]

test_3578 (the song ""Duppatte Ka Palu""), ""Baghban"" (Title Song for Aadesh Shrivastava); ""Soch"" (the song ""Nikal Chali Be"" for Jatin-Lalit); ""Rudraksh"", ""Kal Ho Naa Ho"" (Sad version of the title track for Shankar-Ehsaan-Loy); ""Gangajal"" (Sandesh Shandilya); ""Popcorn Khao Mast Ho Jao"" (Vishal-Shekhar), ""Saawariya"" (Monty Sharma), and ""Om Shanti Om"" (Vishal-Shekhar) and the most popular song for ""Kaante"" (""Mahi Ve"" for Anand Raaj Anand).
test_3578 ""Jhankaar Beats"", ""Lakshya"",""Kaante"", ""Dil Chahta Hai"", ""Kal Ho Na Ho"", ""Hum Tum"", ""Dhoom"", ""Dus"", ""Salaam Namaste"", ""Fight Club - Members Only"",""Koi Mil Gaya"",""Munna Bhai M.B.B.S"", ""Don - The Chase Begins Again"", ""Fanaa"", ""Kabhi Alvida Naa Kehna"", ""Masti (2004 film)"", ""Om Shanti Om"", ""Partner"", ""Welcome"", ""Saawariya"", ""Jab We Met"", ""3 Idiots"", ""Taare Zameen Par"", ""PK"" and ""Prem Ratan Dhan Payo"".
test_3578 in Munna Bhai M.B.B.S."" as Murli Prasad ""Munna"" ""Shahid Kapoor 

100%|██████████| 3610/3610 [01:15<00:00, 47.74it/s]

0
0


In [7]:
# torch.cuda.empty_cache()
# del x

In [8]:
# print(max(sum([len(i) for i in judger.tokenizer(sentences[i+1:i+1+batch_length])['input_ids']]), sum([len(i) for i in judger.tokenizer((batch_length)*[sentences[i]])['input_ids']])))

In [17]:
import timeit
import numpy as np

_k = 16

splitter = get_sentence_splitter()

doc_texts = res[(res.qid == 'test_1038') & (res['rank'] <_k)].docno.apply(lambda x: doc_dict[str(x)]).values
sentences = []
for doc_text in doc_texts:
    sentences += splitter.split(doc_text)

print(f'totally {len(sentences)} sentences')

s = timeit.default_timer()
full_entail_matrix = []
for i in range(len(sentences)-1): # the last one don't need to be calculated
    batch_length = min(16, len(sentences)-i-1)   # only consider the sentences after it
    # print(i, batch_length)
    # print(max(sum([len(j) for j in judger.tokenizer(sentences[i+1:i+1+batch_length])['input_ids']]), sum([len(j) for j in judger.tokenizer((batch_length)*[sentences[i]])['input_ids']])))
    x = judger.get_entailment(sentences[i+1:i+1+batch_length], (batch_length)*[sentences[i]])
    x = (i+1)*[-1] + x + (len(sentences)-i-1-len(x))*[-1]
    full_entail_matrix.append(x)
    torch.cuda.empty_cache()
    # print(x)
full_entail_matrix.append(len(sentences)*[-1])
e = timeit.default_timer()  
print(f'it took {e-s} seconds')
to_check = np.array(full_entail_matrix)
print(to_check.shape)
print(to_check)

totally 82 sentences
it took 31.22659393399954 seconds
(82, 82)
[[-1.00000000e+00  8.45657527e-01  6.71567738e-01 ... -1.00000000e+00
  -1.00000000e+00 -1.00000000e+00]
 [-1.00000000e+00 -1.00000000e+00  3.82292364e-03 ... -1.00000000e+00
  -1.00000000e+00 -1.00000000e+00]
 [-1.00000000e+00 -1.00000000e+00 -1.00000000e+00 ... -1.00000000e+00
  -1.00000000e+00 -1.00000000e+00]
 ...
 [-1.00000000e+00 -1.00000000e+00 -1.00000000e+00 ... -1.00000000e+00
   1.65514718e-03  9.50685237e-04]
 [-1.00000000e+00 -1.00000000e+00 -1.00000000e+00 ... -1.00000000e+00
  -1.00000000e+00  2.54531857e-04]
 [-1.00000000e+00 -1.00000000e+00 -1.00000000e+00 ... -1.00000000e+00
  -1.00000000e+00 -1.00000000e+00]]


In [7]:
# import timeit

# text_0 = 'how is the weather?'
# text_1 = 'really sunny, isn\'t it?'

# s = timeit.default_timer()
# for i in range(100):
#     print(i)
#     x, y = judger.check_implication(50*[text_0], 50*[text_1])
#     del x
#     del y
#     torch.cuda.empty_cache()
# e = timeit.default_timer()  
# print(e-s)

In [ ]:
# torch.cuda.empty_cache()
# del x
# del y